# 🎵 MicroMusicGPT - Colab Pro Scale-Up
This notebook is specifically configured to leverage high-VRAM GPUs (A100/H100) to train a scaled-up version of **MicroMusicGPT**.

### 🧠 Why the Scale-Up?
Unlike text characters (e.g., Tiny Shakespeare), polyphonic MIDI music possesses complex long-range dependencies. A chord requires multiple simultaneous `NOTE_ON` events, separated spatially by `TIME_SHIFT` tokens. If the model forgets a `NOTE_OFF` event, the note sustains indefinitely (hallucination).\n\nBy expanding the architecture from ~10M to ~100M+ parameters and doubling the `block_size` context window to 1024, the model can 'remember' long-range chord progressions and resolve melodic phrases properly.
### 📂 Prerequisite
Before running the training cell, ensure you have uploaded your pre-tokenized dataset tensors to the Colab environment:
- Upload `data/dataset_train.pt` to `/content/data/dataset_train.pt`
- Upload `data/dataset_val.pt` to `/content/data/dataset_val.pt`

In [9]:
!pip install mido pretty_midi matplotlib torch tqdm

In [2]:
!mkdir -p data checkpoints

## 1. Tokenizer Configuration
Here we define our custom MIDI Event Tokenizer. It maps 388 integers to NOTE_ON, NOTE_OFF, and TIME_SHIFT events.

In [13]:
import mido
import torch
from pathlib import Path
try:
    from tqdm import tqdm
except ImportError:
    tqdm = lambda x, **kwargs: x

class MIDITokenizer:
    def __init__(self):
        self.bos_token = 356
        self.eos_token = 357
        self.vocab_size = 388

    def encode(self, midi_path):
        mid = mido.MidiFile(midi_path)
        tokens = [self.bos_token]
        time_buffer_ms = 0

        for msg in mid:
            time_buffer_ms += int(round(msg.time * 1000))

            if msg.type in ['note_on', 'note_off']:
                is_note_on = msg.type == 'note_on' and msg.velocity > 0
                is_note_off = msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0)

                if is_note_on or is_note_off:
                    # Flush accumulated time to sequence
                    while time_buffer_ms >= 10:
                        chunk_ms = min(time_buffer_ms, 1000)
                        chunk_ms = (chunk_ms // 10) * 10 # round down to chunks of 10
                        token = 255 + (chunk_ms // 10)
                        tokens.append(token)
                        time_buffer_ms -= chunk_ms

                    # Emit note bounds
                    if is_note_on:
                        tokens.append(msg.note) # 0-127
                    else:
                        tokens.append(128 + msg.note) # 128-255

        tokens.append(self.eos_token)
        return tokens

    def decode(self, tokens, out_path="demo.mid"):
        # Phase 5 Implementation placeholder
        mid = mido.MidiFile()
        track = mido.MidiTrack()
        mid.tracks.append(track)

        time_buffer_ms = 0
        ticks_per_beat = mid.ticks_per_beat
        tempo = 500000 # 120 bpm default

        for t in tokens:
            if t == self.bos_token or t == self.eos_token:
                continue
            elif 256 <= t <= 355:
                shift_ms = (t - 255) * 10
                time_buffer_ms += shift_ms
            elif 0 <= t <= 127:
                # Convert time_buffer_ms to ticks
                delta_ticks = int(mido.second2tick(time_buffer_ms / 1000.0, ticks_per_beat, tempo))
                track.append(mido.Message('note_on', note=t, velocity=64, time=delta_ticks))
                time_buffer_ms = 0 # reset after emitting event
            elif 128 <= t <= 255:
                note = t - 128
                delta_ticks = int(mido.second2tick(time_buffer_ms / 1000.0, ticks_per_beat, tempo))
                track.append(mido.Message('note_off', note=note, velocity=0, time=delta_ticks))
                time_buffer_ms = 0

        mid.save(out_path)

def build_dataset(raw_dir_path, out_train_path, out_val_path):
    import random
    tokenizer = MIDITokenizer()

    paths = list(Path(raw_dir_path).rglob("*.midi")) + list(Path(raw_dir_path).rglob("*.mid"))
    # Shuffle for rigid split
    random.seed(42)
    random.shuffle(paths)

    split_idx = int(len(paths) * 0.9)
    train_paths = paths[:split_idx]
    val_paths = paths[split_idx:]

    def process_split(split_paths, desc):
        tokens = []
        for p in tqdm(split_paths, desc=desc):
            try:
                enc = tokenizer.encode(str(p))
                tokens.extend(enc)
            except Exception as e:
                print(f"Skipping {p} - Error: {e}")
        return tokens

    train_tokens = process_split(train_paths, "Tokenizing Train")
    val_tokens = process_split(val_paths, "Tokenizing Val")

    print(f"Total train tokens: {len(train_tokens)}")
    print(f"Total val tokens: {len(val_tokens)}")

    torch.save(torch.tensor(train_tokens, dtype=torch.long), out_train_path)
    torch.save(torch.tensor(val_tokens, dtype=torch.long), out_val_path)
    print(f"Saved PyTorch datasets: {out_train_path}, {out_val_path}")

if __name__ == "__main__":
  print("main")
    #build_dataset("data/raw_midi", "data/dataset_train.pt", "data/dataset_val.pt")


main


## 2. Scaled-up GPT Architecture
The `MicroMusicGPTConfig` below has been cranked up:
- `block_size` = 1024
- `n_embd` = 768
- `n_head` = 12
- `n_layer` = 12

*(If you get Cuda Out-Of-Memory on a T4 GPU, lower the `batch_size` to 16 or 32)*

In [15]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from dataclasses import dataclass

@dataclass
class MicroMusicGPTConfig:
    vocab_size: int = 388
    block_size: int = 1024
    batch_size: int = 16  # Reduced from 64 to prevent MPS OOM
    n_embd: int = 768
    n_head: int = 12
    n_layer: int = 12
    dropout: float = 0.2
    device: str = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')

class Head(nn.Module):
    """ one head of self-attention """
    def __init__(self, config, head_size):
        super().__init__()
        self.key = nn.Linear(config.n_embd, head_size, bias=False)
        self.query = nn.Linear(config.n_embd, head_size, bias=False)
        self.value = nn.Linear(config.n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(config.block_size, config.block_size)))
        self.dropout = nn.Dropout(config.dropout)
        self.block_size = config.block_size

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        # compute attention scores
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """
    def __init__(self, config, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(config, head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.ReLU(),
            nn.Linear(4 * config.n_embd, config.n_embd),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """
    def __init__(self, config):
        super().__init__()
        head_size = config.n_embd // config.n_head
        self.sa = MultiHeadAttention(config, config.n_head, head_size)
        self.ffwd = FeedFoward(config)
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.ln2 = nn.LayerNorm(config.n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class MicroMusicGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_embedding_table = nn.Embedding(config.vocab_size, config.n_embd)
        self.position_embedding_table = nn.Embedding(config.block_size, config.n_embd)
        self.blocks = nn.Sequential(*[Block(config) for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.n_embd) # final layer norm
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -self.config.block_size:]
            # get the predictions
            logits, _ = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply temperature
            logits = logits / temperature
            # optionally crop the logits to only the top k options
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx


## 3. Training Loop
This cell will load your uploaded `.pt` tensors and begin optimizing the transformer. We'll track both Cross-Entropy Loss and Perplexity.

In [16]:
import torch
import os
import math
#from src.model import MicroMusicGPT, MicroMusicGPTConfig

def get_batch(train_data, val_data, config, split='train'):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - config.block_size, (config.batch_size,)) # batch_size from config
    x = torch.stack([data[i:i+config.block_size] for i in ix])
    y = torch.stack([data[i+1:i+config.block_size+1] for i in ix])
    x, y = x.to(config.device), y.to(config.device)
    return x, y

@torch.no_grad()
def estimate_loss(model, train_data, val_data, config, eval_iters=50):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(train_data, val_data, config, split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        mean_loss = losses.mean().item()
        perplexity = math.exp(mean_loss)
        out[split] = {'loss': mean_loss, 'perplexity': perplexity}
    model.train()
    return out

def main():
    # Load separate datasets
    train_dataset_path = 'data/dataset_train.pt'
    val_dataset_path = 'data/dataset_val.pt'
    if not os.path.exists(train_dataset_path) or not os.path.exists(val_dataset_path):
        print("Datasets not found. Run tokenizer build script first.")
        return

    train_data = torch.load(train_dataset_path)
    val_data = torch.load(val_dataset_path)
    print(f"Loaded {len(train_data)} train tokens and {len(val_data)} validation tokens.")

    config = MicroMusicGPTConfig()
    print(f"Instantiating model on device: {config.device}")
    model = MicroMusicGPT(config).to(config.device)

    # Check init loss
    xb, yb = get_batch(train_data, val_data, config, 'train')
    _, initial_loss = model(xb, yb)
    print(f"Initial expected loss: {initial_loss.item():.4f}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

    max_iters = 4000
    eval_interval = 400

    os.makedirs('checkpoints', exist_ok=True)

    print("Starting training loop...")
    for iter in range(max_iters):
        if iter % eval_interval == 0 or iter == max_iters - 1:
            metrics = estimate_loss(model, train_data, val_data, config)
            print(f"step {iter}: train cross entropyloss {metrics['train']['loss']:.4f} (perplexity {metrics['train']['perplexity']:.4f}), val loss {metrics['val']['loss']:.4f} (ppl {metrics['val']['perplexity']:.4f})")
            # Serialize checkpoint
            checkpoint_path = f"checkpoints/micromusicgpt_v1_step{iter}.pth"
            torch.save(model.state_dict(), checkpoint_path)

        xb, yb = get_batch(train_data, val_data, config, 'train')
        logits, loss = model(xb, yb)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    final_checkpoint = "checkpoints/micromusicgpt_v1_final.pth"
    torch.save(model.state_dict(), final_checkpoint)
    print(f"Training completed. Final weights saved to {final_checkpoint}")



# Start training!
# Uncomment the line below when datasets are uploaded.
main()

Loaded 3102993 train tokens and 327237 validation tokens.
Instantiating model on device: cuda
Initial expected loss: 6.1940
Starting training loop...
step 0: train cross entropyloss 6.2411 (perplexity 513.4016), val loss 6.2465 (ppl 516.1938)
step 400: train cross entropyloss 3.9064 (perplexity 49.7172), val loss 3.8834 (ppl 48.5906)
step 800: train cross entropyloss 3.2280 (perplexity 25.2283), val loss 3.1950 (ppl 24.4096)
step 1200: train cross entropyloss 2.9312 (perplexity 18.7503), val loss 2.8993 (ppl 18.1612)
step 1600: train cross entropyloss 2.4722 (perplexity 11.8484), val loss 2.4361 (ppl 11.4287)
step 2000: train cross entropyloss 2.1481 (perplexity 8.5690), val loss 2.1548 (ppl 8.6259)
step 2400: train cross entropyloss 1.9131 (perplexity 6.7737), val loss 2.0052 (ppl 7.4279)
step 2800: train cross entropyloss 1.7861 (perplexity 5.9659), val loss 1.9203 (ppl 6.8232)
step 3200: train cross entropyloss 1.6246 (perplexity 5.0763), val loss 1.8392 (ppl 6.2914)
step 3600: trai

## 4. Synthesis & Generation
Once training concludes, use this cell to generate a sequence using Top-K sampling and Temperature scaling to prevent runaway notes.

In [17]:
import torch
import os
#from src.model import MicroMusicGPT, MicroMusicGPTConfig
#from src.tokenizer import MIDITokenizer

def load_model(checkpoint_path):
    config = MicroMusicGPTConfig()
    model = MicroMusicGPT(config)
    if os.path.exists(checkpoint_path):
        model.load_state_dict(torch.load(checkpoint_path, map_location=config.device))
        print(f"Loaded checkpoint from {checkpoint_path}")
    else:
        print("Warning: Checkpoint not found. Generating with uninitialized weights.")

    model.to(config.device)
    model.eval()
    return model, config

def generate_midi(model, config, out_path="demo.mid", max_tokens=1000, seed_tokens=None):
    tokenizer = MIDITokenizer()

    if seed_tokens is None:
        context = torch.tensor([[tokenizer.bos_token]], dtype=torch.long, device=config.device)
    else:
        context = torch.tensor([seed_tokens], dtype=torch.long, device=config.device)

    print("Generating sequence...")
    generated_idx = model.generate(context, max_new_tokens=max_tokens)
    tokens = generated_idx[0].tolist()

    print(f"Decoding {len(tokens)} tokens to {out_path}...")
    tokenizer.decode(tokens, out_path=out_path)
    return out_path

if __name__ == "__main__":
    ckpt = "checkpoints/micromusicgpt_v1_final.pth"
    model, config = load_model(ckpt)
    generate_midi(model, config, out_path="demo.mid", max_tokens=1000)

# Uncomment the line below to generate MIDI after training completion.
# generate_midi('checkpoints/micromusicgpt_v1_final.pth', 'micromusicgpt_colab_demo.mid')

Loaded checkpoint from checkpoints/micromusicgpt_v1_final.pth
Generating sequence...
Decoding 1001 tokens to demo.mid...


In [19]:
def generate_from_ii_V_I(model, config, out_path="demo_chords.mid"):
    tokenizer = MIDITokenizer()

    # 305 is the TIME_SHIFT token for 500ms (255 + 50)
    # D Minor Triad (D4=62, F4=65, A4=69)
    d_min_on   = [62, 65, 69]
    d_min_off  = [62+128, 65+128, 69+128] # Note off is note index + 128

    # G Major Triad (G4=67, B4=71, D5=74)
    g_maj_on   = [67, 71, 74]
    g_maj_off  = [67+128, 71+128, 74+128]

    # C Major Triad (C4=60, E4=64, G4=67)
    c_maj_on   = [60, 64, 67]
    c_maj_off  = [60+128, 64+128, 67+128]

    # Construct the array of tokens manually!
    prompt_tokens = [tokenizer.bos_token] + \
                    d_min_on + [305] + d_min_off + \
                    g_maj_on + [305] + g_maj_off + \
                    c_maj_on + [305] + c_maj_off

    print("Seeding model with manual ii-V-I progression...")
    generate_midi(model, config, out_path=out_path, max_tokens=1000, seed_tokens=prompt_tokens)

# Run it:
generate_from_ii_V_I(model, config, "demo_ii_V_I.mid")


Seeding model with manual ii-V-I progression...
Generating sequence...
Decoding 1022 tokens to demo_ii_V_I.mid...


In [24]:
def generate_from_time_excerpt(model, config, sample_midi_path, max_duration_ms=12000, out_path="demo_time_excerpt_sample2_12s_Seed.mid"):
    tokenizer = MIDITokenizer()
    # Read and encode the real human performance
    real_tokens = tokenizer.encode(sample_midi_path)

    prompt_tokens = [tokenizer.bos_token]
    current_time_ms = 0

    # Iterate through the real tokens until we hit our target duration in milliseconds
    for t in real_tokens[1:]: # skip the original BOS token since we manually added it
        if 256 <= t <= 355:
            # It's a TIME_SHIFT token! Calculate the delay.
            shift_ms = (t - 255) * 10
            current_time_ms += shift_ms

        prompt_tokens.append(t)

        # Stop collecting if we reached our target duration (e.g., 2000ms = 2 seconds)
        if current_time_ms >= max_duration_ms:
            break

    print(f"Extracted first {current_time_ms}ms ({len(prompt_tokens)} tokens) from {sample_midi_path}")

    # Feed this explicit 2-second human intro into the model
    generate_midi(model, config, out_path=out_path, max_tokens=1000, seed_tokens=prompt_tokens)

# Run it (make sure you point it to a valid file in your data/raw_midi/ folder):
generate_from_time_excerpt(model, config, "data/test_beethoven_sample2.midi", max_duration_ms=12000)


Extracted first 12550ms (164 tokens) from data/test_beethoven_sample2.midi
Generating sequence...
Decoding 1164 tokens to demo_time_excerpt_sample2_12s_Seed.mid...


In [22]:
def generate_better_manual_seed(model, config, out_path="demo_rhythmic_chords.mid"):
    tokenizer = MIDITokenizer()

    # TIME_SHIFT tokens: 500ms = 305, 1000ms = 355
    # Let's do Dmin7 -> Gdom7 -> Cmaj7 (held for 2 beats)

    # Dmin7: D4(62), F4(65), A4(69), C5(72)
    dmin7_on  = [62, 65, 69, 72]
    dmin7_off = [n + 128 for n in dmin7_on]

    # Gdom7: G3(55), B3(59), D4(62), F4(65)
    gdom7_on  = [55, 59, 62, 65]
    gdom7_off = [n + 128 for n in gdom7_on]

    # Cmaj7: C4(60), E4(64), G4(67), B4(71)
    cmaj7_on  = [60, 64, 67, 71]
    cmaj7_off = [n + 128 for n in cmaj7_on]

    prompt = [tokenizer.bos_token]

    # Beat 1: play Dmin7 for 500ms
    prompt.extend(dmin7_on)
    prompt.append(305) # wait 500ms
    prompt.extend(dmin7_off)

    # Beat 2: play Gdom7 for 500ms
    prompt.extend(gdom7_on)
    prompt.append(305) # wait 500ms
    prompt.extend(gdom7_off)

    # Beat 3 & 4: play Cmaj7 for 1000ms
    prompt.extend(cmaj7_on)
    prompt.append(355) # wait 1000ms
    prompt.extend(cmaj7_off)

    print(f"Seeding with a rhythmic 1-bar ii-V-I progression ({len(prompt)} tokens)...")
    generate_midi(model, config, out_path=out_path, max_tokens=1000, seed_tokens=prompt)

# Run it:
generate_better_manual_seed(model, config)


Seeding with a rhythmic 1-bar ii-V-I progression (28 tokens)...
Generating sequence...
Decoding 1028 tokens to demo_rhythmic_chords.mid...
